# imports

In [17]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyper Parameters

In [18]:

batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

# Load the data

In [19]:
import os


BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'DATA', 'input.txt')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Loaded {len(text)} characters from {DATA_PATH}")

Loaded 1115394 characters from /Users/I578070/Desktop/NANOGPT/DATA/input.txt


# Encoder and decoder

In [20]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits

In [21]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# Spliting the data into batches

In [22]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Loss

In [23]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Attention Head

In [24]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False) #what info am I looking for
        self.query = nn.Linear(n_embd, head_size, bias=False) # what info do I contain
        self.value = nn.Linear(n_embd, head_size, bias=False) # what info should I provide
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # lower triangular matrix to mask future positions

        self.dropout = nn.Dropout(dropout) # dropout layer to prevent overfitting

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape # B: batch size, T: time steps, C: number of channels (embedding dimension)
        k = self.key(x)   # (B,T,hs) # extract the key
        q = self.query(x) # (B,T,hs) # extract the query
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

# Creating a multihead attention

In [25]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # list of attention heads
        self.proj = nn.Linear(head_size * num_heads, n_embd) # linear projection to combine the outputs of all heads
        self.dropout = nn.Dropout(dropout) # dropout layer to prevent overfitting

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concatenate the outputs of all attention heads along the last dimension
        out = self.dropout(self.proj(out)) # apply linear projection and dropout
        return out

# Feed Forward Layer

In [26]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity and dropout for regularization """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Transformer Block

In [27]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# GPT MODEL

In [28]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # create the emmbeddings 
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # create the positional embeddings
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # stack of transformer blocks
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)
    
    #weight initialization function 
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # forward pass through the model
    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [35]:
model = GPTLanguageModel()
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
m = model.to(device)
print("Selected device:", device)

Selected device: mps


In [36]:
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

10.788929 M parameters
step 0: train loss 4.2831, val loss 4.2758
step 500: train loss 1.7558, val loss 1.9016
step 1000: train loss 1.4017, val loss 1.6088
step 1500: train loss 1.2745, val loss 1.5300
step 2000: train loss 1.1976, val loss 1.5023
step 2500: train loss 1.1347, val loss 1.4875
step 3000: train loss 1.0723, val loss 1.4878
step 3500: train loss 1.0259, val loss 1.4881
step 4000: train loss 0.9701, val loss 1.5079
step 4500: train loss 0.9235, val loss 1.5359
step 4999: train loss 0.8687, val loss 1.5547

Pays.

CLAUDIO:
Ye did conceive the consul leave bethind, even these
Woes words so all.

ISABELLA:
At such deepen sacraminate last the Duke is
Our most report grace:
Your most recommissions have, you suit to
Well please you hath an old of you.

DUKE VINCENTIO:
He'll prove my head to with retire:
And show some hath made fight to have, foul we are
A hard-man's man.

Messenger:
That I had so fond, by what he is gone?

Provost:
Go, and I hope, call him here: I would after y

In [37]:
prompt = "Once upon a time"

# Convert text characters into token IDs
context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

model.eval()

with torch.no_grad():
    generated = m.generate(context, max_new_tokens=200)

print(decode(generated[0].tolist()))

Once upon a time a mock-day.

First Gentleman:
Good Sir, thither: cousin, prepare your majesty.

Second Gentleman:
Faith, I would be counterfeit edious to know me; that we have
wounded to us and that had sings he cam
